# Step 12 — Why is Lateral Meniscus 0.577?

**Settings: Accelerator = `None` (CPU), Internet = `Off`.**

**Inputs:** competition data, `rsna-knee-03-extract-all`, `rsna-knee-08-train-folds`, `rsna-knee-05-train-fold0`

## The problem

Ensemble gold-holdout AUCs:

```
Baker's 0.953   Fracture 0.899   Contusion 0.887   Medial OA 0.876
Lateral OA 0.863   PF OA 0.848   Synovitis 0.823   ACL 0.820
Medial Meniscus 0.800   MCL 0.783   Effusion 0.767
Lateral Meniscus 0.577   <-- 0.27 below everything else
```

Eleven labels average 0.847. One sits at chance. Fixing it is worth about **+0.019** on macro AUC — roughly what the entire 5-fold ensemble bought.

## Four hypotheses, four tests

**H1 — It's noise.** Only 23 positives among 58 gold studies; the standard error on AUC there is around 0.08. Test on the out-of-fold predictions instead, where n = 4,216.

**H2 — Extraction failed.** Reports may say "meniscal tear" without naming a side, so the extractor guesses. If the *labels* are near-random, no model can learn them.

**H3 — The model confuses medial and lateral.** Medial tears are roughly 3x more common, so a model that can't tell the compartments apart defaults toward medial: 0.800 there, chance on lateral. Test whether the medial *prediction* predicts lateral *truth*.

**H4 — Laterality canonicalisation is wrong on some studies.** The `Laterality` DICOM tag was present in only ~79% of series during EDA. For the rest we derive left/right geometrically. If that derivation is wrong, those knees are mirrored and medial/lateral swap.

Each hypothesis implies a different fix, so it's worth 30 minutes of CPU to find out which one it is before spending GPU hours.

### 1 · Load

In [ ]:
import os, glob, warnings
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, pydicom
from sklearn.metrics import roc_auc_score

ROOT = "/kaggle/input/competitions/rsna-knee-abnormality-detection"
def find(name):
    hits = glob.glob(f"/kaggle/input/notebooks/*/*/{name}")
    return hits[0] if hits else None

train  = pd.read_csv(f"{ROOT}/train.csv")
ser    = pd.read_csv(f"{ROOT}/train_series.csv")
pseudo = pd.read_csv(find("pseudo_labels.csv"))
gold   = pd.read_csv(find("gold58.csv"))
oof    = pd.read_csv(find("oof_all.csv"))

LABELS = pd.read_csv(f"{ROOT}/sample_submission.csv").columns[1:].tolist()
KEYS   = ["acl","mcl","medial_meniscus","lateral_meniscus","medial_oa","lateral_oa",
          "pf_oa","effusion","synovitis","bakers","contusion","fracture"]
MED, LAT = "Medial Meniscus", "Lateral Meniscus"

print(f"oof {oof.shape} | pseudo {pseudo.shape} | gold {len(gold)}")

### H1 · Is 0.577 just noise?

The gold holdout has 23 lateral-meniscus positives. On out-of-fold predictions we have 4,216 studies, so the estimate is roughly 8x tighter. If OOF also shows lateral far below medial, the effect is real.

In [ ]:
tp = pseudo.set_index("StudyInstanceUID").loc[oof.StudyInstanceUID, KEYS].values / 100.0
tp = np.nan_to_num(tp, nan=0.2)

print(f"{'label':<18}{'OOF AUC':>9}{'n_pos':>8}")
oof_auc = {}
for i, (k, l) in enumerate(zip(KEYS, LABELS)):
    y = (tp[:, i] > 0.5).astype(int)
    if y.min() == y.max():
        print(f"{l:<18}{'n/a':>9}{y.sum():>8}"); continue
    a = roc_auc_score(y, oof[l].values)
    oof_auc[l] = a
    print(f"{l:<18}{a:>9.3f}{y.sum():>8}")

print(f"\nmacro OOF: {np.mean(list(oof_auc.values())):.3f}")
print(f"Medial Meniscus  {oof_auc.get(MED, float('nan')):.3f}")
print(f"Lateral Meniscus {oof_auc.get(LAT, float('nan')):.3f}")
print(f"gap: {oof_auc.get(MED,0) - oof_auc.get(LAT,0):+.3f}")
print("\n(a gap under ~0.05 means the gold-58 result was mostly noise)")

### H2 · Did extraction fail on this label?

`Lateral Meniscus` extracted at 0.761 against gold, versus 0.882 for medial. Worse, but not chance — so the labels carry real signal and the model should be able to learn them.

This also checks how *decisive* the extractor was. If lateral scores cluster near the "not mentioned" default of 20, the reports simply weren't specifying a side.

In [ ]:
gl = gold["StudyInstanceUID"].tolist()
pg = np.nan_to_num(pseudo.set_index("StudyInstanceUID").loc[gl, KEYS].values.astype(float), nan=20.0)
yg = gold.set_index("StudyInstanceUID").loc[gl, LABELS].values.astype(int)

for name, i in [(MED, KEYS.index("medial_meniscus")), (LAT, KEYS.index("lateral_meniscus"))]:
    a = roc_auc_score(yg[:, LABELS.index(name)], pg[:, i])
    print(f"{name:<18} extraction AUC {a:.3f}")

print("\npseudo-label score distribution (all 4,407):")
for name, k in [(MED,"medial_meniscus"), (LAT,"lateral_meniscus")]:
    v = pseudo[k].dropna()
    near_default = ((v > 12) & (v < 30)).mean()
    print(f"  {name:<18} mean {v.mean():5.1f}  std {v.std():5.1f}  "
          f"frac in 'not mentioned' band (12-30): {near_default:.1%}")
print("\n(a high 'not mentioned' fraction means reports often didn't name the side)")

### H3 · Does the model confuse the two compartments?

The decisive test. If the model's **medial** prediction predicts **lateral** truth about as well as its own lateral prediction does, it isn't distinguishing the compartments — it has learned "meniscal tear somewhere" and attached it to the more common side.

In [ ]:
yl = (tp[:, KEYS.index("lateral_meniscus")] > 0.5).astype(int)
ym = (tp[:, KEYS.index("medial_meniscus")]  > 0.5).astype(int)

print("predicting LATERAL truth with:")
print(f"  lateral prediction : {roc_auc_score(yl, oof[LAT].values):.3f}   <- should be highest")
print(f"  medial  prediction : {roc_auc_score(yl, oof[MED].values):.3f}")
print("\npredicting MEDIAL truth with:")
print(f"  medial  prediction : {roc_auc_score(ym, oof[MED].values):.3f}")
print(f"  lateral prediction : {roc_auc_score(ym, oof[LAT].values):.3f}")
print(f"\ncorrelation between the two predictions: {np.corrcoef(oof[MED], oof[LAT])[0,1]:.3f}")
print("\n(if medial predicts lateral truth as well as lateral does -> H3 confirmed:")
print(" the model sees 'a meniscal tear' but not which side)")

### H4 · Was laterality actually recoverable?

The cache mirrors left knees onto a right-knee convention. It uses the DICOM `Laterality` tag when present and falls back to geometry otherwise. If the tag is missing often, or disagrees between series of the same study, some knees are mirrored wrongly — and medial/lateral swap for exactly those.

This samples 400 studies and checks tag coverage and internal consistency.

In [ ]:
studies = train["StudyInstanceUID"].tolist()
rng = np.random.default_rng(0)
samp = rng.choice(len(studies), 400, replace=False)

have_tag, missing, inconsistent = 0, 0, 0
vals = []
for si in samp:
    suid = studies[si]
    d = f"{ROOT}/train_series/{suid}"
    if not os.path.isdir(d): continue
    lats = set()
    for s in os.listdir(d)[:3]:
        p = os.path.join(d, s)
        fs = os.listdir(p)
        if not fs: continue
        try:
            ds = pydicom.dcmread(os.path.join(p, fs[0]), stop_before_pixels=True)
            l = str(getattr(ds, "Laterality", "") or "").strip().upper()
            if l: lats.add(l)
        except Exception:
            pass
    if not lats:
        missing += 1
    else:
        have_tag += 1
        vals.extend(lats)
        if len(lats) > 1:
            inconsistent += 1

n = have_tag + missing
print(f"studies sampled          : {n}")
print(f"  Laterality tag present : {have_tag} ({have_tag/n:.1%})")
print(f"  tag MISSING            : {missing} ({missing/n:.1%})  <- mirrored by geometry only")
print(f"  tag INCONSISTENT       : {inconsistent} ({inconsistent/max(have_tag,1):.1%} of those with a tag)")
print(f"\nvalues seen: {pd.Series(vals).value_counts().to_dict()}")
print("\n(if >15% missing or any inconsistency, H4 is live: those knees may be mirrored wrongly,")
print(" which swaps medial and lateral for exactly those studies)")

### Verdict

In [ ]:
print("="*66)
print("READ THE RESULTS LIKE THIS")
print("="*66)
print("""
H1  OOF gap small (<0.05)        -> it was noise. Move on to 224px.
H2  lateral extraction near 0.5  -> labels are the problem. Re-prompt the
    or 'not mentioned' band >40%    extractor to force a side, or merge the
                                    two meniscus labels for training and
                                    split them at inference.
H3  medial prediction scores as  -> the model can't tell the compartments
    well on lateral truth           apart. Fix the input, not the labels:
                                    laterality, or resolution (a lateral
                                    meniscus at 160px may be too small).
H4  >15% missing Laterality tag  -> some knees are mirrored wrongly. Derive
    or any inconsistency            laterality from geometry properly, or
                                    drop mirroring entirely and let the model
                                    see left and right knees as they are.
""")